In [1]:
# P1.0 导入库
import numpy as np
import warnings

import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=UserWarning, message=".*normed.*")
import seaborn as sns
from scipy import stats
from scipy.stats import norm
%matplotlib inline

In [2]:
# P1.1 读入训练集
df_train = pd.read_csv('./dataset/train.csv')
df_train.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [3]:
# P1.2 SalePrice的统计特征
df_train['SalePrice'].describe()

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

In [4]:
# P1.3 房价直方图
sns.distplot(df_train['SalePrice']);

C:\Users\32610\AppData\Local\Temp\ipykernel_8756\2509735430.py:2: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  sns.distplot(df_train['SalePrice']);


In [5]:
# P1.3 偏度和峰度
print('偏度: {:.2f}'.format(df_train['SalePrice'].skew()))
print('峰度: {:.2f}'.format(df_train['SalePrice'].kurt()))

偏度: 1.88
峰度: 6.54


In [6]:
# P1.4 GrLivArea 与 SalePrice关系
var = 'GrLivArea'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
data.plot.scatter(x=var, y='SalePrice', ylim=(0,800000));

In [7]:
# P1.4 TotalBsmtSF 与 SalePrice 关系
var = 'TotalBsmtSF'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
data.plot.scatter(x=var, y='SalePrice', ylim=(0,800000));

In [8]:
# P1.4 OverallQual 与 SalePrice关系
var = 'OverallQual'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
f, ax = plt.subplots(figsize=(8, 5))
fig = sns.boxplot(x=var, y="SalePrice", data=data)
fig.axis(ymin=0, ymax=800000);

In [9]:
# P1.4 YearBuilt与SalePrice关系
var = 'YearBuilt'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
f, ax = plt.subplots(figsize=(20, 8))
fig = sns.boxplot(x=var, y="SalePrice", data=data)
fig.axis(ymin=0, ymax=800000);
plt.xticks(rotation=90);

In [10]:
# P1.4相关矩阵
corrmat = df_train.corr()
mask = np.zeros_like(corrmat, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True
f, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(corrmat, mask=mask, cmap="coolwarm", center=0,
            square=True, linewidths=.2, cbar_kws={"shrink":.8});

ValueError: could not convert string to float: 'RL'

In [ ]:
# P1.4 最相关矩阵
k = 10 # 只显示十组变量
cols = corrmat.nlargest(k, 'SalePrice')['SalePrice'].index
cm = np.corrcoef(df_train[cols].values.T)
mask = np.zeros_like(cm, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True
f, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, mask=mask, annot=True, fmt='.2f', cmap="coolwarm", 
            vmax=1, vmin=-1, square=True, linewidths=.2, annot_kws={'size': 12}, 
            yticklabels=cols.values, xticklabels=cols.values);

In [ ]:
# P1.5 缺失数据统计
total = df_train.isnull().sum().sort_values(ascending=False)
percent = (df_train.isnull().sum()/df_train.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data[missing_data.Total>0]

In [ ]:
# P1.5 缺失数据清洗
df_train = df_train.drop((missing_data[missing_data['Total'] > 1]).index,1)
df_train = df_train.drop(df_train.loc[df_train['Electrical'].isnull()].index)
print(df_train.isnull().sum().max()) # 重新统计缺失数据
print(df_train.shape)

In [ ]:
# P1.6 删除GrLivArea离群值
df_train.sort_values(by = 'GrLivArea', ascending = False)[:2]
df_train = df_train.drop(df_train[df_train['Id'] == 1299].index)
df_train = df_train.drop(df_train[df_train['Id'] == 524].index)

In [ ]:
# P1.6 删除TotalBsmtSF离群值
df_train.sort_values(by = 'TotalBsmtSF', ascending = False)[:1]
df_train = df_train.drop(df_train[df_train['Id'] == 333].index)
print(df_train.shape)
df_train_copy = df_train.copy()

In [ ]:
# P1.7 绘制SalePrice的直方图与概率图
sns.distplot(df_train['SalePrice'], fit=norm);
fig = plt.figure()
res = stats.probplot(df_train['SalePrice'], plot=plt)

In [ ]:
# P1.7 对数变换
df_train['SalePrice'] = np.log1p(df_train['SalePrice'])
# 绘制调整后的直方图与概率图
sns.distplot(df_train['SalePrice'], fit=norm);
fig = plt.figure()
res = stats.probplot(df_train['SalePrice'], plot=plt)

In [ ]:
# GrLivArea直方图与概率图
sns.distplot(df_train['GrLivArea'], fit=norm);
fig = plt.figure()
res = stats.probplot(df_train['GrLivArea'], plot=plt)

In [ ]:
# GrLivArea对数变换
df_train['GrLivArea'] = np.log1p(df_train['GrLivArea'])
# GrLivArea对数变换后的直方图与概率图
sns.distplot(df_train['GrLivArea'], fit=norm);
fig = plt.figure()
res = stats.probplot(df_train['GrLivArea'], plot=plt)

In [ ]:
# TotalBsmtSF 的直方图与概率图
sns.distplot(df_train['TotalBsmtSF'], fit=norm);
fig = plt.figure()
res = stats.probplot(df_train['TotalBsmtSF'], plot=plt)

In [ ]:
# P1.8 GrLivArea 与 SalePrice 的残差图
# 对数变换前，GrLivArea 与 SalePrice 的残差图
plt.subplots(figsize = (6,5))
sns.residplot(df_train_copy.GrLivArea, df_train_copy.SalePrice);
# 对数变换后，GrLivArea 与 SalePrice 的残差图
plt.subplots(figsize = (6,5))
sns.residplot(df_train.GrLivArea, df_train.SalePrice);

In [ ]:
# P1.9 手工计算 beta 系数值 
sample_train=df_train.copy()
y_avg = sample_train.SalePrice.mean()
x_avg = sample_train.GrLivArea.mean()
std_y = sample_train.SalePrice.std()
std_x = sample_train.GrLivArea.std()
r_xy = sample_train.corr().loc['GrLivArea','SalePrice']
beta_1 = r_xy*(std_y/std_x)
beta_0 = y_avg - beta_1*x_avg

In [ ]:
# P1.9 计算预测值 y_hat 
sample_train['Yhat'] = beta_0 + beta_1*sample_train['GrLivArea']
sample_train[['SalePrice','Yhat']].head() 

In [ ]:
# P1.10 均方误差(MSE)，计算方法1
print("均方误差(MSE) : {}".
      format(np.square(sample_train['SalePrice'] - sample_train['Yhat']).mean()))

In [ ]:
# P1.10 均方误差(MSE)，计算方法2
from sklearn.metrics import mean_squared_error
mean_squared_error(sample_train['SalePrice'], sample_train.Yhat)